In [1]:
pip install eth_account web3 py-solc-x --break-system-packages

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: /usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
from eth_account import Account

private_key = "2f1a702f4c9b99ceb3ea1b801da326f98dd3c60b201ca01ebcb4fe76d15b5eaf"
account = Account.from_key(private_key)
print("Address:", account.address)


Address: 0x0037cfC76d08C39f3779a183c548F2FF3A2b9165


In [5]:
from web3 import Web3

w3 = Web3(Web3.HTTPProvider("https://eth-sepolia.g.alchemy.com/v2/GBHT7Gy3kAPmnQF3OjZzW"))
print("Connected:", w3.is_connected())
balance = w3.eth.get_balance(account.address)
print("Balance:", w3.from_wei(balance, 'ether'), "ETH")


Connected: True
Balance: 0.048655275793428757 ETH


In [6]:
import solcx
solcx.install_solc('0.8.0')
print("solc 0.8.0 ready")


solc 0.8.0 ready


In [7]:
# --- zfairexchange.sol: Deploy ---
from solcx import compile_source, install_solc

with open('zfairexchange.sol', 'r') as f:
    htlc_source = f.read()

compiled_htlc = compile_source(htlc_source, output_values=['abi', 'bin'], solc_version='0.8.0')
htlc_contract_id, htlc_interface = compiled_htlc.popitem()
htlc_abi = htlc_interface['abi']
htlc_bytecode = htlc_interface['bin']

htlc_contract = w3.eth.contract(abi=htlc_abi, bytecode=htlc_bytecode)
tx = htlc_contract.constructor().build_transaction({
    'from': account.address,
    'nonce': w3.eth.get_transaction_count(account.address),
    'gas': 6000000,
    'gasPrice': w3.eth.gas_price,
    'chainId': 11155111
})
signed_tx = account.sign_transaction(tx)
tx_hash = w3.eth.send_raw_transaction(signed_tx.raw_transaction)
htlc_receipt = w3.eth.wait_for_transaction_receipt(tx_hash)
htlc_address = htlc_receipt.contractAddress
print('HashedTimelockETH deployed at:', htlc_address)


HashedTimelockETH deployed at: 0xF7cFB3FBE766A1Af18218e1498e2E504C231369B


In [8]:
# --- zfairexchange.sol: Call setup() with Waters params from server ---
import requests

resp = requests.get('http://localhost:5000/api/waters_solidity_params?id=1')
sol_params = resp.json()

def g1(p): return (int(p['x']), int(p['y']))
def g2p(p): return ([int(p['x'][0]), int(p['x'][1])], [int(p['y'][0]), int(p['y'][1])])

param_tuple = (
    g1(sol_params['g']),
    g2p(sol_params['g2']),
    g1(sol_params['g1a']),
    g2p(sol_params['g2b']),
    g1(sol_params['u0']),
    [g1(u) for u in sol_params['u_vec']]
)

htlc_instance = w3.eth.contract(address=htlc_address, abi=htlc_abi)
tx = htlc_instance.functions.setup(param_tuple).build_transaction({
    'from': account.address,
    'nonce': w3.eth.get_transaction_count(account.address),
    'gas': 6000000,
    'gasPrice': w3.eth.gas_price,
    'chainId': 11155111
})
signed_tx = account.sign_transaction(tx)
tx_hash = w3.eth.send_raw_transaction(signed_tx.raw_transaction)
receipt = w3.eth.wait_for_transaction_receipt(tx_hash)
print('setup() status:', receipt['status'], '(1 = success)')


setup() status: 1 (1 = success)


In [8]:
# --- zfairexchange.sol: Test full HTLC flow: newContract() -> withdraw() ---
import time

hashlock = 1  # must match id used in waters_solidity_params call above
timelock = int(time.time()) + 3600  # 1 hour from now

# Sender locks ETH in contract
tx = htlc_instance.functions.newContract(
    account.address,
    hashlock,
    timelock
).build_transaction({
    'from': account.address,
    'value': w3.to_wei(0.00001, 'ether'),
    'nonce': w3.eth.get_transaction_count(account.address),
    'gas': 5000000,
    'gasPrice': w3.eth.gas_price,
    'chainId': 11155111
})
signed_tx = account.sign_transaction(tx)
tx_hash = w3.eth.send_raw_transaction(signed_tx.raw_transaction)
receipt = w3.eth.wait_for_transaction_receipt(tx_hash)
print('newContract() status:', receipt['status'], '(1 = success)')

logs = htlc_instance.events.HTLCETHNew().process_receipt(receipt)
contract_id = logs[0]['args']['contractId']
print('Contract ID:', contract_id.hex())

# Receiver reveals preimage (Waters signing key) to withdraw ETH
preimage = (
    g1(sol_params['sk1']),
    g1(sol_params['sk2']),
    g2p(sol_params['sk3'])
)

tx = htlc_instance.functions.withdraw(contract_id, preimage).build_transaction({
    'from': account.address,
    'nonce': w3.eth.get_transaction_count(account.address),
    'gas': 5000000,
    'gasPrice': w3.eth.gas_price,
    'chainId': 11155111
})
signed_tx = account.sign_transaction(tx)
tx_hash = w3.eth.send_raw_transaction(signed_tx.raw_transaction)
receipt = w3.eth.wait_for_transaction_receipt(tx_hash)
print('withdraw() status:', receipt['status'], '(1 = success)')
print('Gas used for withdraw (BN254 pairing):', receipt['gasUsed'])


newContract() status: 1 (1 = success)
Contract ID: 91fd75e57eda95ae50f0c2c08d380559338ecbca9f72994aba7c6fc8f814c185
withdraw() status: 1 (1 = success)
Gas used for withdraw (BN254 pairing): 389301
